### FinGuard-RAG — Experiments v2
  *What this notebook does:*
- Loads + inspects both papers with proper metadata
- Compares chunking strategies
- Embeddings: `bge-base-en-v1.5`
- Vector store: FAISS
- Adds a cross-encoder reranker
- End-to-end pipeline smoke test with Qwen3 8B via Ollama
- Retrieval evaluation: Hit@k and MRR on a gold set

In [1]:
##---LOAD THE SOUCRE---

# langchain-community DeprecationWarning is real — migrate loaders eventually,
# but for now it still works fine for a portfolio project.
from langchain_community.document_loaders import PyPDFLoader

loader_sharc = PyPDFLoader("../data/SHARC.pdf")
loader_gaicf = PyPDFLoader("../data/genai_governance.pdf")

docs_sharc = loader_sharc.load()
docs_gaicf = loader_gaicf.load()

# Tag source paper BEFORE chunking so every chunk inherits it
for d in docs_sharc:
    d.metadata["paper_id"] = "SHARC"
    d.metadata["title"]    = "SHARC: SHAP-Based Interpretability for Regulatory Capital"

for d in docs_gaicf:
    d.metadata["paper_id"] = "GAICF"
    d.metadata["title"]    = "Governing GenAI Across Financial Institutions (SR 26-2)"

all_docs = docs_sharc + docs_gaicf
print(f"SHARC pages : {len(docs_sharc)}")
print(f"GAICF pages : {len(docs_gaicf)}")
print(f"Total pages : {len(all_docs)}")
print("\nSample metadata:", docs_sharc[0].metadata)


C:\Users\Siddharth\AppData\Local\Temp\ipykernel_6324\2050443533.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\Siddharth\FinGuard-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHARC pages : 14
GAICF pages : 16
Total pages : 30

Sample metadata: {'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-07-06T22:00:00+05:30', 'author': 'Ujjwala Vadrevu', 'moddate': '2026-07-06T22:00:00+05:30', 'title': 'SHARC: SHAP-Based Interpretability for Regulatory Capital', 'source': '../data/SHARC.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'paper_id': 'SHARC'}


In [2]:
### CHUNKING ###
## target 800 chars/150 overlap ##
## try tighter chuck later ##
from langchain_text_splitters import RecursiveCharacterTextSplitter

def make_chunks(docs, chunk_size, overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""]  # your good separators, kept
    )
    return splitter.split_documents(docs)

chunks_large  = make_chunks(all_docs, 1500, 300)   # your original
chunks_medium = make_chunks(all_docs, 800,  150)   # recommended

print(f"chunk_size=1500 → {len(chunks_large):4d} chunks  "
      f"(avg {sum(len(c.page_content) for c in chunks_large)//len(chunks_large)} chars)")
print(f"chunk_size=800  → {len(chunks_medium):4d} chunks  "
      f"(avg {sum(len(c.page_content) for c in chunks_medium)//len(chunks_medium)} chars)")
print("\nWe'll use the 800-char chunks going forward.")
chunks = chunks_medium


chunk_size=1500 →   76 chunks  (avg 1220 chars)
chunk_size=800  →  134 chunks  (avg 681 chars)

We'll use the 800-char chunks going forward.


In [3]:
## METADATA ##
# ppr_id,pg,title ##
import random
sample = random.choice(chunks)
print("Text preview :", sample.page_content[:200])
print()
print("Metadata     :", sample.metadata)

# Spot-check: all chunks have paper_id
missing = [c for c in chunks if "paper_id" not in c.metadata]
print(f"\nChunks missing paper_id: {len(missing)}  (expect 0)")


Text preview : governance in consumer credit risk.
3 Generative AI Control Framework (GAICF)
In this section, we introduce the Generative AI Control Framework (GAICF) designed for generative
AI-enabled business work

Metadata     : {'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:8def8d8)', 'creationdate': '', 'author': 'Yiqing Wang; Yixin Kang; Luyun Lin; Siqi Mao', 'doi': 'https://doi.org/10.48550/arXiv.2607.04103', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Governing GenAI Across Financial Institutions (SR 26-2)', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2607.04103v2', 'source': '../data/genai_governance.pdf', 'total_pages': 16, 'page': 4, 'page_label': '4', 'paper_id': 'GAICF'}

Chunks missing paper_id: 0  (expect 0)


In [4]:
## EMBEDDINGS ##
## bge base en v1.5, 768-dim, strong on retrieval benchmarks
from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL = "BAAI/bge-base-en-v1.5"
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    encode_kwargs={"normalize_embeddings": True},   # cosine similarity
)

# Sanity check: embedding dimension
dim = len(embedding_model.embed_query("test"))
print(f"Embedding dim: {dim}")   # expect 768


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5979.41it/s]


Embedding dim: 768


In [5]:
### VECTOR STORE -- FAISS

import os, pickle
from langchain_community.vectorstores import FAISS

FAISS_PATH = "../vectorstore_v2"

if os.path.exists(FAISS_PATH):
    print("Loading existing FAISS index...")
    vectorstore = FAISS.load_local(
        FAISS_PATH, embedding_model, allow_dangerous_deserialization=True
    )
else:
    print(f"Embedding {len(chunks)} chunks and building FAISS index...")
    vectorstore = FAISS.from_documents(chunks, embedding_model)
    vectorstore.save_local(FAISS_PATH)
    # also persist chunks for the reranker (needs raw text)
    with open("../chunks_v2.pkl", "wb") as f:
        pickle.dump(chunks, f)

print(f"FAISS index ready — {vectorstore.index.ntotal} vectors")


Loading existing FAISS index...
FAISS index ready — 134 vectors


In [6]:
### CROSS ENCODER RERANKER
#Pure cosine similarity (`k=5` and stop), Cross-encoder rescores each (query, chunk) pair jointly (slow but accurate) Keep the **top 5** for the LLM
#The cross-encoder sees both query and document together, so it can catch semantic relationships that embedding similarity misses.

from sentence_transformers import CrossEncoder
import numpy as np

RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"   # ~90 MB, fast on CPU
reranker = CrossEncoder(RERANK_MODEL)
print(f"Reranker loaded: {RERANK_MODEL}")

def retrieve_and_rerank(query, k_fetch=20, k_final=5, verbose=True):
    """Two-stage retrieval: dense search → cross-encoder rerank."""
    # Stage 1: dense search
    raw = vectorstore.similarity_search_with_score(
        BGE_QUERY_PREFIX + query, k=k_fetch
    )

    # Stage 2: rerank
    pairs  = [(query, doc.page_content) for doc, _ in raw]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(raw, scores), key=lambda x: x[1], reverse=True)

    results = []
    for (doc, _cos), ce_score in ranked[:k_final]:
        results.append({"doc": doc, "score": float(ce_score)})

    if verbose:
        for i, r in enumerate(results, 1):
            m = r["doc"].metadata
            print(f"[{i}] {m.get('paper_id','?'):5s} | p.{m.get('page','?')} "
                  f"| score={r['score']:+.3f}")
            print("    " + r["doc"].page_content[:150].replace("\n"," ") + "…\n")
    return results


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6455.80it/s]


Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [7]:
### RETRIVAL COMAPARISON
query = "Under stress, does the mean or variance component dominate SVaR?"

print("═"*60)
print("WITHOUT reranker (cosine, k=5):")
print("═"*60)
plain = vectorstore.similarity_search_with_score(BGE_QUERY_PREFIX + query, k=5)
for i, (doc, score) in enumerate(plain, 1):
    m = doc.metadata
    print(f"[{i}] {m.get('paper_id','?'):5s} | p.{m.get('page','?')} | cos={1-score:.3f}")
    print("    " + doc.page_content[:150].replace("\n"," ") + "…\n")

print("═"*60)
print("WITH reranker (fetch 20 → rerank → top 5):")
print("═"*60)
reranked = retrieve_and_rerank(query)


════════════════════════════════════════════════════════════
WITHOUT reranker (cosine, k=5):
════════════════════════════════════════════════════════════
[1] SHARC | p.8 | cos=0.455
    . Variance Dominance: The Core Finding The primary analytical finding of this paper, and its most significant implication for capital limit-setting, i…

[2] SHARC | p.9 | cos=0.446
    daily mean return is already deeply negative, and this directional loss outweighs the additional widening of the distribution driven by increased vari…

[3] SHARC | p.9 | cos=0.430
    . The mean dominance effect is specific to the stressed context, emerging because the scenario conditions the model on extreme directional losses. A S…

[4] SHARC | p.3 | cos=0.407
    This formulation provides a structured decomposition of regulatory capital into baseline, directional, and  volatility-driven components, enabling a t…

[5] SHARC | p.3 | cos=0.389
    role under normal conditions. However, under conditioning on extreme negat

In [8]:
### END TO END RETRIEVAL
## `think=False` tells Qwen3 to skip its `<think>` reasoning blocks. The system prompt instructs it to cite chunk ids inline — those get mapped back to paper + page for display.

import re, requests

OLLAMA_HOST  = "http://localhost:11434"
OLLAMA_MODEL = "qwen3:8b"

SYSTEM = """You are FinGuard, a precise assistant answering questions about AI
governance and ML risk model interpretability in banking, grounded in two papers.

Rules:
- Answer ONLY from the numbered context chunks. No outside knowledge.
- After each claim, add the chunk id in square brackets, e.g. [SHARC-p5-c2].
- If the answer is not in the context, say exactly:
  'That isn't covered in the provided documents.' Do not guess.
- Be concise and use the papers' own terminology."""

def build_context(results):
    blocks = []
    for i, r in enumerate(results):
        doc = r["doc"]
        m   = doc.metadata
        cid = f"{m.get('paper_id','?')}-p{m.get('page','?')}-c{i}"
        blocks.append(f"[{cid}]\n{doc.page_content}")
    return "\n\n".join(blocks)

def strip_thinking(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def ask(question, k_fetch=20, k_final=5):
    results = retrieve_and_rerank(question, k_fetch, k_final, verbose=False)
    context = build_context(results)
    user_msg = f"Context:\n\n{context}\n\nQuestion: {question}\n\nAnswer with inline citations."

    resp = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json={
            "model":   OLLAMA_MODEL,
            "think":   False,
            "stream":  False,
            "options": {"num_predict": 512, "temperature": 0.2},
            "messages":[
                {"role":"system","content":SYSTEM},
                {"role":"user",  "content":user_msg},
            ],
        },
        timeout=180,
    )
    resp.raise_for_status()
    answer = strip_thinking(resp.json()["message"]["content"])
    return answer, results

# ── test it ──
question = "Under stress, does the mean or variance component dominate SVaR?"
print(f"Q: {question}\n")
answer_text, hits = ask(question)
print("A:", answer_text)


Q: Under stress, does the mean or variance component dominate SVaR?



HTTPError: 500 Server Error: Internal Server Error for url: http://localhost:11434/api/chat

In [ ]:
### EXTRACT & CITATIONS
def show_citations(answer_text, results):
    cited = re.findall(r"\[([A-Z]+-p\d+-c\d+)\]", answer_text)
    # map id -> result
    id_map = {
        f"{r['doc'].metadata.get('paper_id','?')}-p{r['doc'].metadata.get('page','?')}-c{i}": r
        for i, r in enumerate(results)
    }
    seen = set()
    print("Cited sources:")
    for cid in cited:
        if cid in id_map and cid not in seen:
            seen.add(cid)
            m = id_map[cid]["doc"].metadata
            print(f"  [{cid}]  {m.get('title','?')[:55]}  |  p.{m.get('page','?')}")
    if not seen:
        print("  (no citations found — model may need more guidance)")

show_citations(answer_text, hits)


In [ ]:
### RETRIEVAL EVALUATION --Hit@k & MRP
## Hit@k :did a chunk from the correct paper reach the top-k?  
## MRR: where did the first correct chunk rank? (1.0 = top, 0.5 = second, …)
from collections import defaultdict
from statistics import mean

GOLD = [
    # (question, expected_paper_id)
    ("What governance gap does SR 26-2 create for generative AI?", "GAICF"),
    ("What does GAICF propose to fill the SR 26-2 gap?",           "GAICF"),
    ("Which supervisory letter does SR 26-2 replace?",             "GAICF"),
    ("What kinds of controls does GAICF define?",                  "GAICF"),
    ("Why is generative AI a governance concern outside model scope?","GAICF"),
    ("What does SHARC stand for?",                                 "SHARC"),
    ("What model does SHARC explain?",                             "SHARC"),
    ("Which SHAP properties does SHARC rely on?",                  "SHARC"),
    ("What risk measure does SHARC decompose?",                    "SHARC"),
    ("Under stress, does mean or variance component dominate SVaR?","SHARC"),
    ("What is Kernel SHAP and why is it used in SHARC?",           "SHARC"),
    ("Name a macro stress scenario used in SHARC.",                "SHARC"),
    ("In which regulatory workflows is SHARC used?",               "SHARC"),
]

def hit_and_mrr(gold, k_fetch=20, k_final=5):
    per = defaultdict(lambda: {"hits":0,"rr":0.0,"n":0})
    rows = []
    for q, paper in gold:
        results = retrieve_and_rerank(q, k_fetch, k_final, verbose=False)
        ranks = [i for i,r in enumerate(results,1)
                 if r["doc"].metadata.get("paper_id")==paper]
        hit = 1 if ranks else 0
        rr  = 1.0/ranks[0] if ranks else 0.0
        per[paper]["hits"] += hit
        per[paper]["rr"]   += rr
        per[paper]["n"]    += 1
        rows.append((paper, q[:55], hit, f"{rr:.2f}"))

    n = sum(p["n"] for p in per.values())
    print(f"{'Paper':<8} {'Hit@k':>6} {'MRR':>6} {'n':>4}")
    print("-"*28)
    for paper, p in per.items():
        print(f"{paper:<8} {p['hits']/p['n']:>6.1%} {p['rr']/p['n']:>6.3f} {p['n']:>4}")
    print("-"*28)
    total_hr  = sum(p['hits'] for p in per.values()) / n
    total_mrr = sum(p['rr']   for p in per.values()) / n
    print(f"{'Overall':<8} {total_hr:>6.1%} {total_mrr:>6.3f} {n:>4}")
    return rows

print(f"Running retrieval eval on {len(GOLD)} questions (k={5})...\n")
rows = hit_and_mrr(GOLD)
